# HARMONY's Australian voice: training

Free Google GPU. Everything is saved to **Google Drive → harmony-voice**, so if Colab disconnects, nothing is lost: just run the cells again and training carries on where it stopped.

**Before you start**
1. Drop **harmony-voice-train.zip** into the **harmony-voice** folder in your Google Drive (Claude made the folder; the zip is in your HARMONY folder under `data\voice-au`).
2. Here: **Runtime → Change runtime type → T4 GPU → Save**.
3. Run the cells one at a time, top to bottom (click ▶ on each, or Shift+Enter).

## 1 · Connect Google Drive and check the GPU

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/harmony-voice'
assert os.path.exists(f'{DRIVE}/harmony-voice-train.zip'), 'Put harmony-voice-train.zip in a Drive folder called harmony-voice first.'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU. Runtime → Change runtime type → T4 GPU, then run this cell again.'
print('Drive connected. GPU:', gpu)

## 2 · Install the voice trainer (about 5 minutes)

In [ ]:
%%bash
set -e
cd /content
if [ ! -d piper1-gpl ]; then
  git clone -q https://github.com/OHF-Voice/piper1-gpl.git
  cd piper1-gpl && git checkout -q 619f513580c04dff5ef9a680f6f3f6979ed94d2e && cd ..
fi
cd piper1-gpl
pip install -q -e '.[train]' onnxscript scikit-build > /tmp/pip.log 2>&1 || (tail -20 /tmp/pip.log; exit 1)
./build_monotonic_align.sh > /tmp/mas.log 2>&1 || (tail -20 /tmp/mas.log; exit 1)
python setup.py build_ext --inplace > /tmp/build.log 2>&1 || (tail -20 /tmp/build.log; exit 1)
python -c "from piper import espeakbridge; from piper.train.vits.monotonic_align import maximum_path; print('Trainer installed.')"

In [ ]:
%%writefile /content/train_piper.py
"""Small wrapper around Piper's trainer (piper1-gpl) that fixes two problems found while testing it.

  python train_piper.py fit ...      same flags as `python -m piper.train fit`
  python train_piper.py export --checkpoint X.ckpt --output-file Y.onnx

1. Piper also keeps checkpoints by a downloaded quality predictor ("val_mos"); when that predictor can't be fetched,
   training crashes after the first epoch. Here checkpoints are kept by validation mel loss only.
2. Recent PyTorch exports ONNX with the new torch.export path by default, which fails on Piper's spline flow.
   Export uses the classic exporter (dynamo=False), which Piper's runtime has always used.
"""
import functools, sys

def fit():
    from lightning.pytorch.callbacks import ModelCheckpoint
    import piper.train.__main__ as piper_train
    piper_train._DEFAULT_CALLBACKS[:] = [ModelCheckpoint(monitor='val_mel', mode='min', save_top_k=3, save_last=True,
                                                         filename='epoch={epoch}-val_mel={val_mel:.4f}', auto_insert_metric_name=False)]
    sys.argv = ['piper.train'] + sys.argv[1:]
    piper_train.main()

def export():
    import torch
    torch.onnx.export = functools.partial(torch.onnx.export, dynamo=False)
    import piper.train.export_onnx as piper_export
    sys.argv = ['piper.train.export_onnx'] + sys.argv[2:]
    piper_export.main()

if __name__ == '__main__':
    if len(sys.argv) > 1 and sys.argv[1] == 'export': export()
    else: fit()

## 3 · Unpack the recordings and fetch the starting model

In [ ]:
import json, os, subprocess
if not os.path.exists('/content/data/train/metadata.csv'):
    subprocess.run(['unzip', '-q', '-o', f'{DRIVE}/harmony-voice-train.zip', '-d', '/content/data'], check=True)
speakers = json.load(open('/content/data/train/speakers.json'))['speakers']
print(f"{sum(speakers.values())} clips from {len(speakers)} women; HARMONY's voice has {speakers['harmony']}.")

from huggingface_hub import hf_hub_download
# VCTK (medium): a 109-speaker English Piper model. Starting from it means HARMONY only has to learn the
# Australian accent and these voices, not how to speak from scratch.
BASE = hf_hub_download('rhasspy/piper-checkpoints', 'en/en_GB/vctk/medium/epoch=545-step=1511328.ckpt', repo_type='dataset')
print('Starting model ready.')

## 4 · Train

Starts training in the background and shows progress. **You can stop this cell any time (■ button) without stopping the training**: run it again to watch progress, or go to step 5 to listen.

Each "epoch" is one pass over all the recordings. On a T4 expect very roughly 1 minute per epoch. It saves to Drive every epoch, and stops by itself after 400.

In [ ]:
import glob, os, subprocess, time
RUN = f'{DRIVE}/run'
os.makedirs(RUN, exist_ok=True)
running = subprocess.run(['pgrep', '-f', 'train_piper.py fit'], capture_output=True).stdout.strip()
if not running:
    last = sorted(glob.glob(f'{RUN}/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
    start = ['--ckpt_path', last[-1]] if last else ['--model.warmstart_ckpt', BASE]
    print('Carrying on from', last[-1]) if last else print('Starting fresh from the VCTK model.')
    cmd = ['python', '/content/train_piper.py', 'fit',
           '--data.voice_name', 'en_AU-harmony', '--data.csv_path', '/content/data/train/metadata.csv',
           '--data.audio_dir', '/content/data/train/wavs', '--data.espeak_voice', 'en-gb-x-rp',
           '--data.cache_dir', '/content/cache', '--data.config_path', f'{DRIVE}/en_AU-harmony-medium.onnx.json',
           '--data.batch_size', '16', '--data.num_workers', '2', '--model.sample_rate', '22050',
           '--model.num_speakers', str(len(speakers)), '--trainer.max_epochs', '400',
           '--trainer.accelerator', 'gpu', '--trainer.default_root_dir', RUN] + start
    log = open('/content/train.log', 'a')
    subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd='/content/piper1-gpl')
    time.sleep(5)

import re
seen = 0
try:
    while True:
        text = open('/content/train.log', errors='ignore').read()
        for line in text[seen:].splitlines():
            if re.search(r'warmstart|Traceback|Error|out of memory|Epoch \d+:\s+100%', line):
                print(line[-160:])
        seen = len(text)
        if not subprocess.run(['pgrep', '-f', 'train_piper.py fit'], capture_output=True).stdout.strip():
            print('Training stopped. If that was unexpected, send Claude the last lines above.'); break
        time.sleep(30)
except KeyboardInterrupt:
    print('Stopped watching. Training is still running.')

## 5 · Listen

Run this whenever you like (even while training) to hear the latest checkpoint. Early on she'll sound robotic or muffled; it gets better with every epoch. Tell Claude what you hear.

In [ ]:
import sys; sys.path.insert(0, '/content/piper1-gpl/src')  # Piper was installed after this notebook started
import glob, json, os, subprocess, wave
from IPython.display import Audio, display
ckpts = sorted(glob.glob(f'{RUN}/lightning_logs/*/checkpoints/*.ckpt'), key=os.path.getmtime)
assert ckpts, 'No checkpoint yet. Wait for the first epoch to finish.'
latest = [c for c in ckpts if c.endswith('last.ckpt')] or ckpts
onnx = '/content/preview.onnx'
r = subprocess.run(['python', '/content/train_piper.py', 'export', '--checkpoint', latest[-1], '--output-file', onnx],
                   capture_output=True, text=True, cwd='/content/piper1-gpl')
assert r.returncode == 0, r.stderr[-2000:]
subprocess.run(['cp', f'{DRIVE}/en_AU-harmony-medium.onnx.json', onnx + '.json'], check=True)
from piper import PiperVoice
from piper.config import SynthesisConfig
voice = PiperVoice.load(onnx)
lines = ["G'day! I'm Harmony. How can I help?", "Got it. I'll remind you to pick up the kids at half past four.",
         "No worries at all. One thing at a time, we've got this."]
print('Checkpoint:', os.path.basename(latest[-1]))
for i, text in enumerate(lines):
    path = f'/content/sample{i}.wav'
    with wave.open(path, 'wb') as w:
        voice.synthesize_wav(text, w, syn_config=SynthesisConfig(speaker_id=voice.config.speaker_id_map['harmony'], noise_scale=0.33, noise_w_scale=0.4))
    print(text); display(Audio(path))

## 6 · Save the finished voice

When she sounds right, run this. It puts **en_AU-harmony-medium.onnx** (and its .json) in your Drive's harmony-voice folder. Then tell Claude: Claude installs it into HARMONY.

In [ ]:
import shutil, subprocess
ckpts = sorted(glob.glob(f'{RUN}/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
out = f'{DRIVE}/en_AU-harmony-medium.onnx'
r = subprocess.run(['python', '/content/train_piper.py', 'export', '--checkpoint', ckpts[-1], '--output-file', '/content/final.onnx'],
                   capture_output=True, text=True, cwd='/content/piper1-gpl')
assert r.returncode == 0, r.stderr[-2000:]
shutil.copy('/content/final.onnx', out)
print('Saved to Google Drive → harmony-voice → en_AU-harmony-medium.onnx (+ .onnx.json). Tell Claude it is done.')